# ==============================================================================
# 🇻🇳 KAGGLE BENCHMARK CHUYÊN BIỆT: CUSTOMTOOLS-VI (E0 -> E4)
# ==============================================================================
Notebook này dùng để đánh giá riêng tập dữ liệu **CustomTools-VI** cho các checkpoint từ E0 đến E4:
- **`test_seen.jsonl`** (800 mẫu, 400 pos / 400 neg): Đánh giá trên các công cụ đã xuất hiện trong train (seen tools).
- **`test_unseen.jsonl`** (800 mẫu, 400 pos / 400 neg): Đánh giá khả năng **Zero-shot generalization** trên các công cụ hoàn toàn mới.

**Điểm đặc biệt**:
1. Chạy tuần tự từng adapter (`E0`, `E1`, `E2`, `E3`, `E4` hoặc chỉ chọn 1 subset bất kỳ).
2. Sử dụng cơ chế Subprocess đa GPU: Mỗi khi xong 1 experiment/subset, tiến trình tự kết thúc và **giải phóng 100% VRAM GPU** về 0 MB trước khi nạp adapter tiếp theo.
3. Tự động chấm điểm và xuất **Bảng so sánh tổng hợp đối chiếu E0 -> E4** ngay tại chỗ.

In [ ]:
import os
import json
from pathlib import Path

# ==============================================================================
# ⚙️ CẤU HÌNH DANH SÁCH EXPERIMENTS CẦN TEST
# ==============================================================================
# Bạn có thể chọn chạy toàn bộ ["e0", "e1", "e2", "e3", "e4"] 
# hoặc chỉ chạy những experiment bạn muốn, ví dụ: ["e3"], ["e4"], hoặc ["e0", "e3", "e4"]
EXPERIMENTS_TO_RUN = ["e0", "e1", "e2", "e3", "e4"]

# Chọn kích thước model (2B hoặc 4B):
MODEL_ID = "unsloth/Qwen3.5-2B"  # Hoặc "unsloth/Qwen3.5-4B"
HF_NAME = "ThinhDao"

# Tự động nhận diện đường dẫn dữ liệu trên Kaggle
candidate_paths = [
    Path("/kaggle/input/datasets/phcthnho/tool-calling-vi-experiments"),
    Path("/kaggle/input/tool-calling-vi-experiments"),
]
DATA_ROOT = next((p for p in candidate_paths if p.is_dir()), candidate_paths[0])
CUSTOM_DIR = DATA_ROOT / "custom_vi"

WORKING_DIR = Path("/kaggle/working/custom_benchmarks")
WORKING_DIR.mkdir(parents=True, exist_ok=True)

os.environ["DATA_ROOT"] = str(DATA_ROOT)
os.environ["CUSTOM_DIR"] = str(CUSTOM_DIR)
os.environ["WORKING_DIR"] = str(WORKING_DIR)
os.environ["MODEL_ID"] = MODEL_ID
os.environ["HF_NAME"] = HF_NAME

print("="*60)
print(f"EXPERIMENTS SẼ CHẠY : {EXPERIMENTS_TO_RUN}")
print(f"MODEL_ID            : {MODEL_ID}")
print(f"HF_NAME             : {HF_NAME}")
print(f"CUSTOM_DIR          : {CUSTOM_DIR}")
print(f"WORKING_DIR         : {WORKING_DIR}")
print("="*60)

def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as source:
        return [json.loads(line) for line in source if line.strip()]

In [ ]:
assert CUSTOM_DIR.is_dir(), f"Không tìm thấy thư mục: {CUSTOM_DIR}. Hãy kiểm tra dataset Kaggle."

for subset in ["test_seen", "test_unseen"]:
    path = CUSTOM_DIR / f"{subset}.jsonl"
    assert path.is_file(), f"Thiếu file: {path}"
    records = read_jsonl(path)
    assert len(records) == 800, f"Số mẫu không khớp: {len(records)} != 800 tại {path.name}"
    pos = sum(1 for r in records if r.get("function_calls"))
    neg = sum(1 for r in records if not r.get("function_calls"))
    print(f"  ✓ {subset:<12}: {len(records)} mẫu (Positive: {pos}, Negative: {neg})")

print("\n✅ Preflight Data Check PASS! Sẵn sàng benchmark CustomTools-VI.")

In [ ]:
%pip install -q -U peft "transformers>=5.2.0" "accelerate>=1.0" "bitsandbytes>=0.43"

In [ ]:
import torch

assert torch.cuda.is_available(), "Vui lòng bật GPU accelerator trên Kaggle (khuyên dùng 2x T4)"
for index in range(torch.cuda.device_count()):
    print(f"GPU {index}: {torch.cuda.get_device_name(index)}")

In [ ]:
%%writefile worker_custom.py
import os
import sys
import json
import time
from pathlib import Path
import torch
from transformers import AutoModelForImageTextToText, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

gpu_id = int(sys.argv[1])
total_gpus = int(sys.argv[2])
subset = sys.argv[3] if len(sys.argv) > 3 else "test_seen"  # "test_seen" hoặc "test_unseen"
exp_id = sys.argv[4].lower() if len(sys.argv) > 4 else "e0"

MODEL_ID = os.environ.get("MODEL_ID", "unsloth/Qwen3.5-2B")
HF_NAME = os.environ.get("HF_NAME", "ThinhDao")
MODEL_NAME_TAG = MODEL_ID.rsplit('/', 1)[-1]

# Nếu là e0: Chạy Zero-shot trực tiếp từ base model, không nạp adapter
if exp_id == "e0":
    ADAPTER_ID = None
    RUN_NAME = f"e0_{MODEL_NAME_TAG.lower()}"
else:
    ADAPTER_ID = f"{HF_NAME}/{MODEL_NAME_TAG}_{exp_id.upper()}"
    RUN_NAME = f"{exp_id}_{MODEL_NAME_TAG.lower()}"

WORKING_DIR = Path(os.environ.get("WORKING_DIR", "/kaggle/working/custom_benchmarks"))
RUN_DIR = WORKING_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

candidate_paths = [
    Path("/kaggle/input/datasets/phcthnho/tool-calling-vi-experiments"),
    Path("/kaggle/input/tool-calling-vi-experiments"),
]
default_data_root = next((p for p in candidate_paths if p.is_dir()), candidate_paths[0])
CUSTOM_DIR = Path(os.environ.get("CUSTOM_DIR", str(default_data_root / "custom_vi")))

PART_PATH = RUN_DIR / f"eval_predictions_{subset}_part_{gpu_id}.jsonl"

BATCH_SIZE = 16
MAX_NEW_TOKENS = 128
SYSTEM_PROMPT_VI = "Bạn là trợ lý AI có khả năng sử dụng công cụ."

def format_tool(tool: dict) -> dict:
    return {"type": "function", "function": {"name": tool["name"], "description": tool.get("description", ""), "parameters": tool.get("parameters", {})}}

def format_call(call: dict) -> dict:
    return {"type": "function", "function": {"name": call["name"], "arguments": call.get("arguments", {})}}

def native_row(record: dict) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_VI},
        {"role": "user", "content": record["query"]},
    ]
    calls = [format_call(call) for call in record.get("function_calls", [])]
    if calls:
        messages.append({"role": "assistant", "content": "", "tool_calls": calls})
    else:
        fallback = "Hiện tại tôi chưa thể thực hiện yêu cầu này."
        messages.append({"role": "assistant", "content": record.get("assistant_content") or fallback})
    return {"id": record["id"], "messages": messages, "tools": [format_tool(tool) for tool in record.get("tools", [])]}

def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as s:
        return [json.loads(line) for line in s if line.strip()]

test_records = read_jsonl(CUSTOM_DIR / f"{subset}.jsonl")
my_records = test_records[gpu_id::total_gpus]

completed_ids = set()
if PART_PATH.exists():
    with PART_PATH.open(encoding="utf-8") as s:
        completed_ids = {json.loads(line)["id"] for line in s if line.strip()}

pending_records = [r for r in my_records if r["id"] not in completed_ids]
print(f"[{exp_id.upper()} | {subset.upper()} | GPU {gpu_id}] Tổng mẫu: {len(my_records)} | Đã xong: {len(completed_ids)} | Cần chạy: {len(pending_records)}")

if pending_records:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(
        ADAPTER_ID if ADAPTER_ID else MODEL_ID,
        trust_remote_code=True,
        padding_side="left",
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"[{exp_id.upper()} | GPU {gpu_id}] Loading base model {MODEL_ID} on cuda:{gpu_id}...")
    try:
        base_model = AutoModelForImageTextToText.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            device_map={"": gpu_id},
            dtype=torch.float16,
            trust_remote_code=True,
        )
    except Exception as e:
        print(f"[{exp_id.upper()} | GPU {gpu_id}] Fallback to AutoModelForCausalLM due to: {e}")
        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            device_map={"": gpu_id},
            dtype=torch.float16,
            trust_remote_code=True,
        )

    if ADAPTER_ID:
        print(f"[{exp_id.upper()} | GPU {gpu_id}] Loading adapter from {ADAPTER_ID}...")
        model = PeftModel.from_pretrained(base_model, ADAPTER_ID).eval()
    else:
        print(f"[{exp_id.upper()} | GPU {gpu_id}] Running base model zero-shot (No adapter)...")
        model = base_model.eval()

    def prompt_text(record: dict) -> str:
        row = native_row(record)
        return tokenizer.apply_chat_template(row["messages"][:-1], tools=row["tools"], tokenize=False, add_generation_prompt=True, enable_thinking=False)

    pending_items = [{"record": r, "prompt": prompt_text(r)} for r in pending_records]
    pending_items.sort(key=lambda x: len(x["prompt"]))

    with PART_PATH.open("a", encoding="utf-8") as output_file:
        for start in range(0, len(pending_items), BATCH_SIZE):
            batch_items = pending_items[start : start + BATCH_SIZE]
            batch_records = [item["record"] for item in batch_items]
            prompts = [item["prompt"] for item in batch_items]

            encoded = tokenizer(prompts, padding=True, truncation=True, max_length=4096, add_special_tokens=False, return_tensors="pt").to(f"cuda:{gpu_id}")
            t0 = time.perf_counter()
            with torch.inference_mode():
                generated = model.generate(
                    **encoded,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )
            batch_latency_ms = (time.perf_counter() - t0) * 1000.0
            generated_tokens = generated[:, encoded["input_ids"].shape[1] :]
            texts = tokenizer.batch_decode(generated_tokens, skip_special_tokens=False)

            for record, raw_output in zip(batch_records, texts, strict=True):
                output_file.write(json.dumps({
                    "id": record["id"],
                    "query": record["query"],
                    "gold": record.get("function_calls", []),
                    "raw_output": raw_output,
                    "batch_latency_ms": round(batch_latency_ms, 2),
                    "batch_size": len(batch_items),
                }, ensure_ascii=False) + "\n")
            output_file.flush()

            done = len(completed_ids) + start + len(batch_items)
            if done % 100 == 0 or done == len(my_records):
                print(f"[{exp_id.upper()} | {subset.upper()} | GPU {gpu_id}] Xử lý: {done}/{len(my_records)} ({done/len(my_records)*100:.1f}%) | Latency: {batch_latency_ms:.0f}ms")

print(f"[{exp_id.upper()} | {subset.upper()} | GPU {gpu_id}] ✅ HOÀN THÀNH!")


In [ ]:
import re
import json

TOOL_RE = re.compile(
    r"<tool_call>\s*<function\s*=\s*([^>\s]+)\s*>(.*?)</function>\s*</tool_call>",
    re.DOTALL | re.IGNORECASE,
)
PARAM_RE = re.compile(
    r"<parameter\s*=\s*([^>\s]+)\s*>(.*?)</parameter>",
    re.DOTALL | re.IGNORECASE,
)
JSON_CALL_RE = re.compile(
    r"<tool_call>\s*(\{.*?\})\s*</tool_call>",
    re.DOTALL | re.IGNORECASE,
)

def parse_native_output(text: str) -> tuple[list[dict], list[str]]:
    calls = []
    errors = []
    open_tags = len(re.findall(r"<tool_call\b", text, re.IGNORECASE))
    close_tags = len(re.findall(r"</tool_call\s*>", text, re.IGNORECASE))
    if open_tags != close_tags:
        errors.append("unbalanced_tool_call_tags")
        
    for name, body in TOOL_RE.findall(text):
        arguments = {}
        for parameter, value in PARAM_RE.findall(body):
            value = value.strip()
            try:
                arguments[parameter.strip()] = json.loads(value)
            except (json.JSONDecodeError, TypeError):
                arguments[parameter.strip()] = value
        calls.append({"name": name.strip(), "arguments": arguments})
        
    if not calls:
        for json_str in JSON_CALL_RE.findall(text):
            try:
                parsed = json.loads(json_str.strip())
                if isinstance(parsed, dict) and "name" in parsed:
                    calls.append({
                        "name": parsed["name"],
                        "arguments": parsed.get("arguments", {}),
                    })
            except Exception:
                pass

    if not calls and open_tags > 0:
        errors.append("malformed_tool_call")
    return calls, errors
print("✅ Helper parser sẵn sàng.")


In [ ]:
import torch
import gc
import subprocess

all_experiments_results = {}
CUSTOM_SUBSETS = ["test_seen", "test_unseen"]

for exp_id in EXPERIMENTS_TO_RUN:
    exp_name = exp_id.lower()
    model_tag = MODEL_ID.rsplit('/', 1)[-1].lower()
    run_name = f"{exp_name}_{model_tag}"
    run_dir = WORKING_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    
    print("
" + "="*70)
    print(f"🎯 BẮT ĐẦU BENCHMARK CUSTOM CHO: {exp_id.upper()} ({run_name})")
    print("="*70)
    
    all_experiments_results[exp_id.upper()] = {}

    for subset in CUSTOM_SUBSETS:
        print(f"
🚀 Đang chạy suy luận: {exp_id.upper()} -> {subset.upper()}")
        # Chạy worker bằng subprocess: khi kết thúc, GPU VRAM tự động giải phóng 100%
        !python worker_custom.py 0 2 {subset} {exp_id} & python worker_custom.py 1 2 {subset} {exp_id} & wait
        
        # Dọn dẹp bộ nhớ phụ trợ
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
        # Gộp file kết quả từ 2 GPU
        predictions_by_id = {}
        for gpu_id in range(torch.cuda.device_count()):
            part_path = run_dir / f"eval_predictions_{subset}_part_{gpu_id}.jsonl"
            if part_path.exists():
                with part_path.open(encoding="utf-8") as f:
                    for line in f:
                        if line.strip():
                            item = json.loads(line)
                            predictions_by_id[item["id"]] = item

        test_records = read_jsonl(CUSTOM_DIR / f"{subset}.jsonl")
        pred_path = run_dir / f"eval_predictions_{run_name}_{subset}.jsonl"
        with pred_path.open("w", encoding="utf-8") as output_file:
            for record in test_records:
                rec_id = record["id"]
                if rec_id in predictions_by_id:
                    output_file.write(json.dumps(predictions_by_id[rec_id], ensure_ascii=False) + "\n")
        print(f"  ✓ Đã gộp {len(predictions_by_id)}/{len(test_records)} mẫu vào: {pred_path.name}")

        # Chấm điểm chi tiết
        rows = read_jsonl(pred_path)
        expected_ids = {r["id"] for r in test_records}
        is_complete = len(rows) == len(expected_ids)

        positive = negative = tool_correct = negative_correct = exact_match = syntax_errors = 0
        latency_ms = 0.0
        scored_rows = []
        for row in rows:
            predicted, errors = parse_native_output(row["raw_output"])
            gold = row["gold"]
            is_positive = bool(gold)
            tool_match = [call["name"] for call in predicted] == [call["name"] for call in gold]
            exact = predicted == gold
            if is_positive:
                positive += 1
                tool_correct += tool_match
            else:
                negative += 1
                negative_correct += not predicted and not errors
            exact_match += exact
            syntax_errors += bool(errors)
            latency_ms += row["batch_latency_ms"] / row["batch_size"]
            scored_rows.append({
                **row,
                "predicted": predicted,
                "errors": errors,
                "tool_match": tool_match if is_positive else (not predicted and not errors),
                "exact_match": exact,
            })

        metrics = {
            "experiment": exp_id.upper(),
            "subset": subset,
            "total_samples": len(rows),
            "is_complete": is_complete,
            "tool_accuracy_pos_pct": round(100 * tool_correct / positive, 2) if positive else None,
            "non_fc_recall_pct": round(100 * negative_correct / negative, 2) if negative else None,
            "arga_exact_match_pct": round(100 * exact_match / len(rows), 2) if rows else 0.0,
            "syntax_error_rate_pct": round(100 * syntax_errors / len(rows), 2) if rows else 0.0,
            "avg_latency_ms": round(latency_ms / len(rows), 2) if rows else 0.0,
        }
        all_experiments_results[exp_id.upper()][subset] = metrics

        (run_dir / f"eval_predictions_{run_name}_{subset}_scored.json").write_text(
            json.dumps(scored_rows, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
        )
        (run_dir / f"eval_metrics_{run_name}_{subset}.json").write_text(
            json.dumps(metrics, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
        )
        print(f"    -> {subset.upper()}: Tool Acc = {metrics['tool_accuracy_pos_pct']}%, Exact Match (ArgA) = {metrics['arga_exact_match_pct']}%, Non-FC Recall = {metrics['non_fc_recall_pct']}%")

print("
" + "="*70)
print("🎉 ĐÃ HOÀN THÀNH ĐÁNH GIÁ TOÀN BỘ CÁC EXPERIMENT ĐƯỢC CHỌN!")
print("="*70)


In [ ]:
# In bảng tổng hợp so sánh đa mô hình trên CustomTools-VI
print("
" + "="*85)
print(f"🏆 BẢNG TỔNG HỢP SO SÁNH ĐỐI CHIẾU TRÊN BỘ DỮ LIỆU CUSTOMTOOLS-VI ({MODEL_ID})")
print("="*85)

header = f"{'Experiment':<12} | {'Seen Acc (%)':<14} | {'Seen ArgA (%)':<14} | {'Unseen Acc (%)':<16} | {'Unseen ArgA (%)':<16} | {'ArgA Gap':<10}"
print(header)
print("-" * len(header))

for exp_key in EXPERIMENTS_TO_RUN:
    exp_u = exp_key.upper()
    if exp_u in all_experiments_results:
        res = all_experiments_results[exp_u]
        seen = res.get("test_seen", {})
        unseen = res.get("test_unseen", {})
        
        s_acc = seen.get("tool_accuracy_pos_pct", "N/A")
        s_arga = seen.get("arga_exact_match_pct", "N/A")
        u_acc = unseen.get("tool_accuracy_pos_pct", "N/A")
        u_arga = unseen.get("arga_exact_match_pct", "N/A")
        
        gap_str = "N/A"
        if isinstance(s_arga, (int, float)) and isinstance(u_arga, (int, float)):
            gap_str = f"{s_arga - u_arga:+.2f}%"
            
        s_acc_str = f"{s_acc}%" if s_acc != "N/A" else "N/A"
        s_arga_str = f"{s_arga}%" if s_arga != "N/A" else "N/A"
        u_acc_str = f"{u_acc}%" if u_acc != "N/A" else "N/A"
        u_arga_str = f"{u_arga}%" if u_arga != "N/A" else "N/A"
        
        row = f"{exp_u:<12} | {s_acc_str:<14} | {s_arga_str:<14} | {u_acc_str:<16} | {u_arga_str:<16} | {gap_str:<10}"
        print(row)

print("="*85)
print("Ghi chú:")
print(" - Seen Acc / ArgA  : Năng lực trên công cụ đã huấn luyện (chỉ E4 được train có chủ đích).")
print(" - Unseen Acc / ArgA: Năng lực Zero-shot generalization trên công cụ hoàn toàn mới.")
print(" - ArgA Gap         : Độ chênh lệch giữa Seen và Unseen (càng nhỏ càng tổng quát hóa tốt).")
